# Introduction to Machine Learning -- MC886A/MO444A
# Assignment 3 - Neural Networks, Inductive Bias and Distribution Shift

**Professor:** Marcelo da Silva Reis

**PED:** Heigon Soldera

In [ ]:
# Enter your name and RA number below.

print("Lucas Treviso Bandeira", "|", "322979")
print("Mariana Dourado Ximenes de Sena Santos", "|", "300643")

---

## Objective

In Assignment 2 you saw that the model selected by cross-validation on the training distribution is not
necessarily the most robust one when the test distribution changes. There, the assumption we varied was the
**penalty** of a linear model. In this assignment we vary the **architecture**.

A multilayer perceptron (MLP) and a convolutional neural network (CNN) can both fit a training set almost
perfectly, but they do it under different assumptions about the input, their **inductive biases**:
- an MLP gives every pixel position its own weight and knows nothing about which pixels are neighbors;
- a CNN detects local patterns with small filters that are shared across all positions, and combines them
  hierarchically.

When several hypotheses explain the training data equally well, these assumptions decide which one the network
learns, and the difference only shows up when the test distribution changes. For example, a classifier of
histopathology images can recognize a type of tissue from the shape and the arrangement of its cells, which are the
same in every hospital, or from the exact colors produced by the staining protocol of the hospital where the
training images were acquired. Both solutions fit the training set, but only one of them survives a new hospital.

The goal of this assignment is to test, with controlled experiments, which structural assumptions and which training
choices help a neural network generalize to a clinical center it has never seen.

### Guidelines

> Please read these guidelines carefully.

1. Answers can be written in English or Portuguese.

2. Use data visualization as a primary tool for understanding. Inspect the training data, the transformed inputs, and
   model behavior. When an assumption is uncertain, verify it empirically through plots.

3. The provided code serves as a reference. Modifications are allowed, including changes to functions and the
   introduction of new components. Code clarity and organization are required. The hands-on of Lecture 18 (Deep
   Learning with PyTorch) contains reference implementations of the training loop, of an MLP, of a CNN and of data
   augmentation.

4. We recommend running this notebook on a GPU, for example on Google Colab (GPU runtime) or on a local GPU.

5. The use of generative AI tools (e.g., ChatGPT, GitHub Copilot, DeepSeek, LLaMA) is permitted strictly for
   **implementation support**. Their use for interpreting, analyzing, or discussing experimental results is
   discouraged. Submissions must include:
   - A declaration of which models were used.
   - The prompts that generated any incorporated code.

6. Do not hesitate to reach out to the PED/PAD via Discord.

### Experiment structure

Each experiment must follow the structure below:

1. **Question**: the empirical question guiding the exploration.
2. **Hypothesis**: a falsifiable answer to the question that makes a concrete, measurable prediction.
3. **Design**: justify why the proposed experiment can test the hypothesis. Focus on the reasoning, not on how the
   code works.
4. **Results**: run the experiment and report the metrics.
5. **Discussion**: interpret the results in terms of the hypothesis. Explicitly state whether the results support or
   contradict it, and why.

A hypothesis must be **falsifiable**: the proposed experiment must be capable of providing evidence *against* it.
Statements such as "the model is complex enough" or "the features are informative" do not qualify.

A negative result is not a failure: a well-reasoned experiment that contradicts the hypothesis is graded as highly as
one that confirms it, provided the interpretation is honest and coherent.

For Experiment 1, the question and hypothesis are given. Your task is to complete the missing code, write the design
justification, and discuss the results.
For Experiments 2-4, only the question is given. You must state the hypothesis and discuss the results; in
Experiments 2 and 3 you also justify the design, and in Experiment 4 you choose and evaluate the final model.

### Evaluation protocol

Throughout this assignment, **ID** (*in-distribution*) refers to patches from the same clinical center as the training
set, and **OOD** (*out-of-distribution*) to patches from a different center, which the model never sees during
training.

- **Model selection** (number of epochs, hyperparameters, early stopping, choice of the final model) uses only the
  **ID validation** split.
- The **OOD validation** split is used to *measure* the generalization gap in Experiments 1-4, never to choose
  hyperparameters or models.
- The **OOD test** split is used **only once**, at the end of Experiment 4. Do not look at its images or metrics
  before that.

In [ ]:
import copy
import os
import random
import time
import urllib.request

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
from IPython.display import display

# ── Reproducibility ────────────────────────────────────────────────────────────
# Do not change this seed. Re-run the notebook from scratch before submitting.
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True   # reproducible results on the GPU
torch.backends.cudnn.benchmark = False

DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

---

## Dataset — PathMNIST

Histopathology is the study of thin sections of tissue under a microscope, often to detect disease. The most common
technique stains the section with hematoxylin and eosin (H&E): hematoxylin colors the cell nuclei purple-blue, and
eosin colors the cytoplasm and the extracellular matrix pink, so the pathologist can tell the structures apart.

We use **PathMNIST**, from the [MedMNIST v2](https://medmnist.com/) benchmark, in its 64×64 version. Each sample is a
64×64 RGB patch of an H&E-stained section of colorectal tissue, labeled with one of **9 tissue classes**:

| Label | Tissue |
|---|---|
| 0 | adipose |
| 1 | background |
| 2 | debris |
| 3 | lymphocytes |
| 4 | mucus |
| 5 | smooth muscle |
| 6 | normal colon mucosa |
| 7 | cancer-associated stroma |
| 8 | colorectal adenocarcinoma epithelium |

The patches come from two datasets of the same study (Kather et al., 2019):
- the training and validation patches come from **NCT-CRC-HE-100K**;
- the test patches come from **CRC-VAL-HE-7K**, acquired at a **different clinical center**, with other patients.

Each center has its own staining protocol, scanner and population of patients, so the patches of the new center
differ from the training ones in ways that have nothing to do with the tissue class.

We use the following splits:

| Split | Source | Patches | Role in this assignment |
|---|---|---|---|
| `train` | NCT-CRC-HE-100K | 89,996 | Training |
| `id_val` | NCT-CRC-HE-100K | 10,004 | **ID validation**: early stopping and model selection |
| `ood_val` | CRC-VAL-HE-7K, first half | 3,590 | **OOD validation**: measures the generalization gap |
| `ood_test` | CRC-VAL-HE-7K, second half | 3,590 | **OOD test**: used only once, at the end of Experiment 4 |

The `id_val` patches were drawn at random from the same dataset as the training patches. The official test set of
PathMNIST is split in two halves, stratified by class, with the seed of this notebook. The two halves may contain
patches of the same patients, so they are not fully independent: `ood_test` protects the final result against
choices made *because* a model did well on `ood_val`, but not against everything the two halves have in common.

**Metrics.** We report the accuracy and the **balanced accuracy**, the mean of the recalls of the 9 classes
(`sklearn.metrics.balanced_accuracy_score`). Unlike the accuracy, the balanced accuracy does not change when only
the proportions of the classes change, which will matter in Experiment 4. The **ID–OOD gap**, the balanced accuracy
on `id_val` minus the balanced accuracy on `ood_val`, is the main measure of generalization to the new center.

The next cell downloads the 64×64 version of PathMNIST (about 1 GB, only once) from the
[MedMNIST repository on Zenodo](https://zenodo.org/records/10519652). Alternatively, you can download the data of this
assignment from our Google Classroom page: place the file `pathmnist_64.npz` in a folder named `data`, inside the
working directory of the notebook (on Colab, `/content/data`), and the cell will not download it again. You can also
install the `medmnist` package and use `medmnist.PathMNIST(split=..., size=64, download=True)`.

In [ ]:
DATA_DIR = "data"
DATA_PATH = os.path.join(DATA_DIR, "pathmnist_64.npz")
DATA_URL = "https://zenodo.org/records/10519652/files/pathmnist_64.npz"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(DATA_PATH):                    # download only once
    print("Downloading PathMNIST 64x64 (about 1 GB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

CLASSES = ["adipose", "background", "debris", "lymphocytes", "mucus", "smooth muscle",
           "normal colon mucosa", "cancer-associated stroma", "colorectal adenocarcinoma epithelium"]

with np.load(DATA_PATH) as f:
    # Images stay as uint8 (N, 3, 64, 64) to save memory; they are converted to float one batch at a time.
    X_train = torch.from_numpy(f["train_images"]).permute(0, 3, 1, 2).contiguous()
    y_train = torch.from_numpy(f["train_labels"]).long().squeeze(1)
    X_id_val = torch.from_numpy(f["val_images"]).permute(0, 3, 1, 2).contiguous()
    y_id_val = torch.from_numpy(f["val_labels"]).long().squeeze(1)
    X_external = torch.from_numpy(f["test_images"]).permute(0, 3, 1, 2).contiguous()
    y_external = torch.from_numpy(f["test_labels"]).long().squeeze(1)

In [ ]:
def stratified_split(y: torch.Tensor, fraction: float, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    # Splits the indices of y in two parts, with `fraction` of each class in the first part.
    first, second = [], []
    for c in np.unique(y.numpy()):
        idx = rng.permutation(np.flatnonzero(y.numpy() == c))
        cut = round(fraction * len(idx))
        first.append(idx[:cut])
        second.append(idx[cut:])
    return np.sort(np.concatenate(first)), np.sort(np.concatenate(second))


rng = np.random.default_rng(seed)

# The external center is split in two halves: OOD validation and OOD test.
ood_val_idx, ood_test_idx = stratified_split(y_external, 0.5, rng)
X_ood_val, y_ood_val = X_external[ood_val_idx], y_external[ood_val_idx]
X_ood_test, y_ood_test = X_external[ood_test_idx], y_external[ood_test_idx]
del X_external, y_external

SPLITS = {
    "train": (X_train, y_train),
    "id_val": (X_id_val, y_id_val),
    "ood_val": (X_ood_val, y_ood_val),
    "ood_test": (X_ood_test, y_ood_test),   # do not use before the end of Experiment 4
}
for name, (X, y) in SPLITS.items():
    print(f"{name:8s}: {tuple(X.shape)}, {X.dtype}")

### Preprocessing

The networks of this assignment are trained from scratch, so we standardize each color channel with the mean and the
standard deviation of the **training split only** (no leakage), and apply the same statistics to every split.

`Preprocess` turns a batch of uint8 images into the model input, on the device. It also accepts a `color_fn`, a
transformation of the colors applied before the standardization (Experiment 3), and, during training, an `augment`
function applied to the RGB images in [0, 1] (Experiments 2 and 3).

In [ ]:
def to_unit(x: torch.Tensor) -> torch.Tensor:
    # uint8 batch (N, 3, H, W) -> float in [0, 1], on the device.
    return x.to(DEVICE, non_blocking=True).float().div(255)


class Preprocess:
    # uint8 batch -> [augmentation] -> [color transformation] -> standardization with training statistics.

    def __init__(self, color_fn=None):
        self.color_fn = color_fn if color_fn is not None else (lambda x: x)
        total, total_sq, n_pixels = 0.0, 0.0, 0
        for i in range(0, len(X_train), 5_000):          # statistics of the TRAINING split only
            x = self.color_fn(to_unit(X_train[i:i + 5_000])).double()
            total = total + x.sum(dim=(0, 2, 3))
            total_sq = total_sq + (x ** 2).sum(dim=(0, 2, 3))
            n_pixels += x.shape[0] * x.shape[2] * x.shape[3]
        mean = total / n_pixels
        self.mean = mean.float().view(1, -1, 1, 1)
        self.std = (total_sq / n_pixels - mean ** 2).sqrt().float().view(1, -1, 1, 1)

    def __call__(self, x: torch.Tensor, augment=None) -> torch.Tensor:
        x = to_unit(x)
        if augment is not None:                          # training only
            x = augment(x)
        return (self.color_fn(x) - self.mean) / self.std


rgb = Preprocess()
print("channel mean:", rgb.mean.flatten().cpu().numpy().round(3), "| channel std:", rgb.std.flatten().cpu().numpy().round(3))

---

## Data Exploration

Before starting the experiments, take a moment to explore the dataset and understand its structure. Below, we show
random patches of each class, from the training center and from the new center (OOD validation).

The goal is to have a better idea of the task, and of what problems are already recognizable without training any
model: class balance, differences between the two centers, which information is available in a patch, which
transformations do not change the class.

**Do not look at the OOD test split (`ood_test`) in this section.**

In [ ]:
n = 4
rng = np.random.default_rng(seed)

fig, axs = plt.subplots(len(CLASSES), 2 * n, figsize=(2 * n * 1.4, len(CLASSES) * 1.45), constrained_layout=True)
for c, name in enumerate(CLASSES):
    for j, split in enumerate(["train", "ood_val"]):
        X, y = SPLITS[split]
        for k, i in enumerate(rng.choice(np.flatnonzero(y.numpy() == c), size=n, replace=False)):
            ax = axs[c, j * n + k]
            ax.imshow(X[i].permute(1, 2, 0).numpy())
            ax.set_xticks([])
            ax.set_yticks([])
            if c == 0:
                ax.set_title("training center" if split == "train" else "new center", fontsize=9)
    axs[c, 0].set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")

fig.suptitle("Random patches of each class: training center (left) and new center, OOD validation (right)", fontsize=12)
plt.show()

In [ ]:
# Shared setup of the exploration below. Only train, id_val and ood_val are used

EDA_SPLITS = [
    "train",
    "id_val",
    "ood_val"
]

SPLIT_COLORS = {
    "train": "#2a78d6",
    "id_val": "#1baf7a",
    "ood_val": "#eb6834"
}

SPLIT_LABELS = {
    "train": "train (training center)",
    "id_val": "id_val (training center)",
    "ood_val": "ood_val (new center)"
}

SHORT_CLASSES = [
    "adipose",
    "background",
    "debris",
    "lymphocytes",
    "mucus",
    "smooth\nmuscle",
    "normal\nmucosa",
    "stroma",
    "tumor\nepithelium"
]

labels = {s: SPLITS[s][1].numpy() for s in EDA_SPLITS}
patch_means = {s: SPLITS[s][0].numpy().mean(axis=(2, 3)) / 255 for s in EDA_SPLITS}   # mean color of each patch: (N, 3)


def plot_per_class(features: dict[str, np.ndarray], names: list[str], title: str) -> None:
    # One row per feature: its distribution in each class, training center vs new center
    # (boxes: quartiles and median; whiskers: 5th and 95th percentiles)
    fig, axs = plt.subplots(len(names), 1, figsize=(14, 2.8 * len(names)), sharex=True)
    pos, centers = np.arange(len(CLASSES)), ["train", "ood_val"]
    for f, (ax, name) in enumerate(zip(axs, names)):
        for k, s in enumerate(centers):
            data = [features[s][labels[s] == c, f] for c in range(len(CLASSES))]
            ax.boxplot(data, positions=pos + 0.36 * (k - 0.5), widths=0.3, whis=(5, 95), showfliers=False,
                       patch_artist=True, boxprops=dict(facecolor=SPLIT_COLORS[s]), medianprops=dict(color="black"))
        ax.set_ylabel(name)
        ax.grid(axis="y", alpha=0.3)
    axs[-1].set_xticks(pos, SHORT_CLASSES)
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=SPLIT_COLORS[s], edgecolor="black") for s in centers]
    axs[0].legend(handles, [SPLIT_LABELS[s] for s in centers], loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2,
                  frameon=False)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# PLOT: class proportions in each split

counts = pd.DataFrame({s: np.bincount(labels[s], minlength=len(CLASSES)) for s in EDA_SPLITS}, index=CLASSES)
props = 100 * counts / counts.sum()

fig, ax = plt.subplots(figsize=(10, 5.5))
pos = np.arange(len(CLASSES))
for k, s in enumerate(EDA_SPLITS):
    ax.barh(pos + 0.27 * (k - 1), props[s], height=0.25, color=SPLIT_COLORS[s],
            label=f"{SPLIT_LABELS[s]}, n = {counts[s].sum():,}")
ax.axvline(100 / len(CLASSES), color="gray", linewidth=1)
ax.set_yticks(pos, CLASSES)
ax.invert_yaxis()
ax.set_xlabel("patches of the split (%); gray line = balanced split, 1/9 = 11.1%")
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)
fig.suptitle("Class proportions in each split", fontsize=12)
plt.tight_layout()
plt.show()

display(props.round(1).assign(**{"ood_val / train": (props["ood_val"] / props["train"]).round(2)}))


In [ ]:
# PLOT: mean intensity of each color channel of a patch, per class and center

plot_per_class(patch_means, ["mean red", "mean green", "mean blue"],
               "Mean intensity of each color channel of a patch, per class and center")

In [ ]:
# PLOT: mean amount of hematoxylin and eosin of a patch, per class and center
# Color deconvolution of skimage.color.rgb2hed, the same used by stain_jitter in Experiment 3: the optical density
# -log(intensity) of each pixel is a linear combination of the concentrations of hematoxylin, eosin and DAB

hed_from_rgb = np.linalg.inv([[0.65, 0.70, 0.29],    # hematoxylin
                              [0.07, 0.99, 0.11],    # eosin
                              [0.27, 0.57, 0.78]])   # DAB (not used in H&E staining)


def mean_optical_density(X: torch.Tensor, chunk: int = 5_000) -> np.ndarray:
    # Mean optical density of each color channel of every patch: (N, 3)
    return torch.cat([-torch.log(to_unit(X[i:i + chunk]).clamp(min=1e-6)).mean(dim=(2, 3)).cpu()
                      for i in range(0, len(X), chunk)]).numpy()


stains = {s: mean_optical_density(SPLITS[s][0]) @ hed_from_rgb for s in ["train", "ood_val"]}
plot_per_class(stains, ["hematoxylin\n(nuclei)", "eosin\n(cytoplasm, fibers)"],
               "Mean concentration of each stain in a patch, per class and center")

# Relative change of the median concentration of each stain in the new center
display(pd.DataFrame({stain: [np.median(stains["ood_val"][labels["ood_val"] == c, k]) /
                              np.median(stains["train"][labels["train"] == c, k]) for c in range(len(CLASSES))]
                      for k, stain in enumerate(["hematoxylin: ood_val / train", "eosin: ood_val / train"])},
                     index=CLASSES).round(2))

In [ ]:
# PLOT: average patch of each class in each center

fig, axs = plt.subplots(2, len(CLASSES), figsize=(1.75 * len(CLASSES), 4.2), constrained_layout=True)
for r, s in enumerate(["train", "ood_val"]):
    X, y = SPLITS[s]
    for c in range(len(CLASSES)):
        axs[r, c].imshow(X[y == c].numpy().mean(axis=0).transpose(1, 2, 0) / 255)
        axs[r, c].set_xticks([])
        axs[r, c].set_yticks([])
        axs[0, c].set_title(SHORT_CLASSES[c], fontsize=9)
    axs[r, 0].set_ylabel(SPLIT_LABELS[s].replace(" (", "\n("), fontsize=9)
fig.suptitle("Average patch of each class in each center", fontsize=12)
plt.show()

In [ ]:
# PLOT: telling the training center from the new center using only the mean color of a patch
# Train is subsampled with the same number of patches of each class as ood_val
# Each repetition draws a new subsample and new folds

N_REPEATS = 10


def center_scores(colors_train: np.ndarray, colors_ood: np.ndarray, cv_seed: int) -> np.ndarray:
    # Balanced accuracy on each of the 5 folds of a logistic regression that tells train (0) from ood_val (1)
    X = np.vstack([colors_train, colors_ood])
    y = np.r_[np.zeros(len(colors_train)), np.ones(len(colors_ood))]
    cv = StratifiedKFold(5, shuffle=True, random_state=cv_seed)
    clf = make_pipeline(StandardScaler(), LogisticRegression())
    return cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy")


ood_counts = np.bincount(labels["ood_val"], minlength=len(CLASSES))
scores = {name: [] for name in ["all classes", *CLASSES]}
for rep in range(N_REPEATS):
    rng = np.random.default_rng(seed + rep)
    matched = np.concatenate([rng.choice(np.flatnonzero(labels["train"] == c), size=n, replace=False)
                              for c, n in enumerate(ood_counts)])
    colors, classes = patch_means["train"][matched], labels["train"][matched]
    scores["all classes"].append(center_scores(colors, patch_means["ood_val"], seed + rep))
    for c, name in enumerate(CLASSES):
        ood_colors = patch_means["ood_val"][labels["ood_val"] == c]
        scores[name].append(center_scores(colors[classes == c], ood_colors, seed + rep))

center_df = pd.DataFrame({"bal_acc_mean": [np.mean(v) for v in scores.values()],
                          "bal_acc_std": [np.std(v) for v in scores.values()],
                          "patches_per_center": [ood_counts.sum(), *ood_counts]}, index=list(scores))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(center_df.index, center_df["bal_acc_mean"] - 0.5, left=0.5, xerr=center_df["bal_acc_std"], capsize=3,
        color=SPLIT_COLORS["train"])
ax.axhline(0.5, color="gray", linewidth=1)
ax.set_xlim(0.5, 1.0)
ax.invert_yaxis()
ax.set_xlabel(f"balanced accuracy, {N_REPEATS} subsamples x 5-fold CV (0.5 = chance; bars: mean ± std)")
ax.grid(axis="x", alpha=0.3)
ax.set_title("Telling the training center from the new center using only the mean color of a patch", fontsize=12)
plt.tight_layout()
plt.show()

display(center_df.round(3))

In [ ]:
# PLOT: recognizing the tissue class using only the mean color of a patch, in each center
# A logistic regression trained on the mean color of the train patches, evaluated on both validation splits

color_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1_000))
color_clf.fit(patch_means["train"], labels["train"])
recalls = pd.DataFrame({s: confusion_matrix(labels[s], color_clf.predict(patch_means[s]), normalize="true").diagonal()
                        for s in ["id_val", "ood_val"]}, index=CLASSES)

fig, ax = plt.subplots(figsize=(10, 5.5))
pos = np.arange(len(CLASSES))
for k, s in enumerate(recalls):
    ax.barh(pos + 0.36 * (k - 0.5), recalls[s], height=0.34, color=SPLIT_COLORS[s],
            label=f"{SPLIT_LABELS[s]}: balanced accuracy {recalls[s].mean():.2f}")
ax.axvline(1 / len(CLASSES), color="gray", linewidth=1)
ax.set_yticks(pos, CLASSES)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("recall of the class (gray line = chance, 1/9)")
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
fig.suptitle("Recognizing the tissue class using only the mean color of a patch", fontsize=12)
plt.tight_layout()
plt.show()

display(recalls.round(2))

> **Question.** (1.0 pt)
>
> What are your findings? Describe the task, the balance of the classes, and how the two centers differ. Which
> properties of the patches seem to be stable across centers, and which change from one center to the other?
>
> Based only on this exploration, predict which architecture, MLP or CNN, is more likely to rely on properties that
> change between centers, and why.

A tarefa consiste em classificar cada patch RGB de 64 × 64 pixels de uma lâmina histológica colorretal em uma das nove classes do PathMNIST. No treino e na validação do mesmo centro (ID), as proporções das classes são praticamente idênticas e variam de 8,8% a 14,3%. Na validação do novo centro (OOD), a distribuição é mais desigual, com a proporção de tecido adiposo subindo de 10,4% no treino para 18,6%, enquanto a de detritos cai de 11,5% para 4,7%.

Algumas características morfológicas parecem se preservar nos dois centros. Nos patches de tecido adiposo, observam-se grandes espaços claros delimitados por bordas finas. Nos de linfócitos, distinguem-se agrupamentos densos de pequenos núcleos e, nos de músculo liso, feixes alongados em diferentes orientações. Ao comparar amostras da mesma classe entre os dois centros, porém, observam-se diferenças de coloração. No novo centro, a mediana estimada de hematoxilina é 35% maior nos detritos e 34% menor no tecido adiposo em relação ao treino.

Essas diferenças ajudam a identificar o centro de origem, mas sua intensidade e direção dependem da classe. Quando as classes são analisadas separadamente, uma regressão logística que usa apenas a cor média RGB alcança acurácia balanceada de 59,3% no estroma a 91,3% no fundo, com 80,9% no tecido adiposo. Quando as nove classes são reunidas, mantendo a mesma proporção de cada classe nos dois centros, a acurácia balanceada fica em 54,3%, próxima dos 50% esperados ao acaso. Isso sugere que uma diferença de cor perceptível para determinado tecido pode ser pouco informativa sobre o centro de origem quando se consideram todos os tecidos em conjunto.

A cor média também ajuda a prever a classe do tecido. Usando apenas os três valores médios de RGB, o classificador alcança 52% de acurácia balanceada em ID e 53% em OOD, acima dos 11,1% esperados ao acaso para nove classes. A proximidade dessas médias esconde mudanças importantes por classe. Do centro de treino para o novo centro, o recall do músculo liso cai de 66% para 42%, enquanto o da classe de fundo sobe de 57% para 100%.

Com base nessa exploração, nossa hipótese é que a MLP seja mais propensa que a CNN a depender da tonalidade global. Ao calcular a média dos patches de cada classe, obtemos imagens quase uniformes. Isso sugere que estruturas como bordas e núcleos aparecem em posições variadas e se diluem na média, enquanto a tonalidade global permanece visível. Como a MLP atribui pesos próprios a cada posição, ela precisa aprender a reconhecer uma estrutura em diferentes regiões da imagem. Agregar as intensidades dos pixels oferece uma alternativa mais simples, e os 52% de acurácia balanceada em ID mostram que essa informação já é útil para prever o tecido durante o treino. A CNN, por compartilhar filtros entre posições, pode reconhecer a mesma borda ou textura em diferentes regiões. Por isso, esperamos que ela aproveite melhor os padrões morfológicos que parecem se preservar entre os centros, embora também possa usar a cor.

---

## Training utilities

All the experiments train and evaluate models with the same four functions, which you will implement in the next
cell. The hands-on of Lecture 18 contains a training loop that you can start from.

> **Implementation.** (1.0 pt)
>
> Implement the four functions below, keeping their names, arguments and return values, since the rest of the notebook
> uses them:
>
> - `make_loader(split, batch_size, shuffle, shuffle_seed)` returns a `DataLoader` over the patches and labels of
>   `SPLITS[split]`. When `shuffle=True`, the order must be **reproducible**: shuffle with a `torch.Generator` seeded
>   with `shuffle_seed`.
> - `predict(model, loader, preprocess, input_fn=None)` returns the predicted classes and the true labels of a whole
>   loader, as two NumPy arrays. Each batch goes through `preprocess` and then, if it is given, through `input_fn`
>   (used in Experiment 2). The model must be in evaluation mode, and no gradients are needed.
> - `evaluate(model, loader, preprocess, input_fn=None)` returns `{"acc": ..., "bal_acc": ...}`, the accuracy and the
>   balanced accuracy of the predictions of `predict`.
> - `fit(model, train_loader, val_loader, optimizer, n_epochs, preprocess, augment=None, verbose=True)` trains the
>   model with `nn.CrossEntropyLoss` (all models output **9 logits** per patch, and the loss applies the softmax).
>   Each training batch goes through `preprocess(x, augment)`. After every epoch, `fit` evaluates the model on
>   `val_loader` and keeps a copy of the weights of the epoch with the best **ID validation balanced accuracy** (early
>   stopping). At the end it restores those weights, so the model used afterwards is the one of the best epoch. It
>   returns a dictionary `history` with:
>   - `"train_loss"`, `"train_acc"`, `"val_acc"`, `"val_bal_acc"`: lists with one value per epoch;
>   - `"best_epoch"`: the epoch whose weights were kept, counting from 1;
>   - `"train_time"`: the total training time, in seconds.
>
>   With `verbose=True`, it prints the metrics of each epoch.
>
> Model selection inside `fit` must use only the ID validation split. Run the sanity check below before starting the
> experiments.

The cell after your implementation is already complete:
- `report` evaluates a trained model on the ID and the OOD validation splits and stores the results in `RESULTS`,
  which you will use in the Summary table;
- `plot_histories` plots the histories returned by `fit`.

In [ ]:
def make_loader(split: str, batch_size: int = 512, shuffle: bool = False, shuffle_seed: int = seed) -> DataLoader:
    # DataLoader over the uint8 patches and labels of SPLITS[split]. The shuffled order is fixed by shuffle_seed.
    X, y = SPLITS[split]
    generator = torch.Generator().manual_seed(shuffle_seed) if shuffle else None
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle, generator=generator,
                      pin_memory=(DEVICE == "cuda"))


def predict(model: nn.Module, loader: DataLoader, preprocess: Preprocess, input_fn=None) -> tuple[np.ndarray, np.ndarray]:
    # Predicted classes (argmax of the logits) and true labels of the whole loader, as NumPy arrays.
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = preprocess(x)
            if input_fn is not None:
                x = input_fn(x)
            all_preds.append(model(x).argmax(dim=1).cpu())
            all_labels.append(y)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def evaluate(model: nn.Module, loader: DataLoader, preprocess: Preprocess, input_fn=None) -> dict[str, float]:
    # Accuracy and balanced accuracy (mean of the per-class recalls) of the predictions of predict().
    preds, labels = predict(model, loader, preprocess, input_fn)
    return {"acc": float((preds == labels).mean()), "bal_acc": float(balanced_accuracy_score(labels, preds))}


def fit(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, optimizer: optim.Optimizer,
        n_epochs: int, preprocess: Preprocess, augment=None, verbose: bool = True) -> dict:
    # Trains with cross-entropy and keeps the weights of the epoch with the best ID validation balanced accuracy.
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_bal_acc": [], "best_epoch": None,
               "train_time": None}
    best_bal_acc, best_state = -1.0, None
    start = time.time()

    for epoch in range(1, n_epochs + 1):
        model.train()
        loss_sum, n_correct, n_seen = 0.0, 0, 0
        for x, y in train_loader:
            x = preprocess(x, augment)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            # Weighted by the batch size, so that a smaller last batch does not distort the epoch average.
            loss_sum += loss.item() * len(y)
            n_correct += (logits.argmax(dim=1) == y).sum().item()
            n_seen += len(y)

        val = evaluate(model, val_loader, preprocess)
        history["train_loss"].append(loss_sum / n_seen)
        history["train_acc"].append(n_correct / n_seen)
        history["val_acc"].append(val["acc"])
        history["val_bal_acc"].append(val["bal_acc"])

        if val["bal_acc"] > best_bal_acc:                 # early stopping on the ID validation only
            best_bal_acc = val["bal_acc"]
            best_state = copy.deepcopy(model.state_dict())
            history["best_epoch"] = epoch

        if verbose:
            print(f"epoch {epoch:2d}/{n_epochs} | train loss {history['train_loss'][-1]:.4f} "
                  f"| train acc {history['train_acc'][-1]:.3f} | val acc {val['acc']:.3f} "
                  f"| val bal. acc {val['bal_acc']:.3f}{'  *' if history['best_epoch'] == epoch else ''}")

    model.load_state_dict(best_state)
    history["train_time"] = time.time() - start
    if verbose:
        print(f"best epoch: {history['best_epoch']} (val bal. acc {best_bal_acc:.3f}) "
              f"| training time: {history['train_time']:.0f} s")
    return history

In [ ]:
RESULTS: dict[str, dict] = {}   # one row per trained model, filled by report()
MODELS: dict[str, tuple] = {}   # name -> (model, Preprocess used to evaluate it)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def report(name: str, model: nn.Module, preprocess: Preprocess, history: dict | None = None) -> dict:
    # ID and OOD validation metrics of a trained model, stored in RESULTS.
    id_m = evaluate(model, make_loader("id_val"), preprocess)
    ood_m = evaluate(model, make_loader("ood_val"), preprocess)
    RESULTS[name] = {
        "params": count_params(model),
        "train_time_s": round(history["train_time"]) if history else None,
        "id_acc": id_m["acc"], "id_bal_acc": id_m["bal_acc"],
        "ood_acc": ood_m["acc"], "ood_bal_acc": ood_m["bal_acc"],
        "gap_bal_acc": id_m["bal_acc"] - ood_m["bal_acc"],
    }
    MODELS[name] = (model, preprocess)
    r = RESULTS[name]
    print(f"{name}: ID bal. acc {r['id_bal_acc']:.3f} | OOD bal. acc {r['ood_bal_acc']:.3f} | gap {r['gap_bal_acc']:+.3f} "
          f"| ID acc {r['id_acc']:.3f} | OOD acc {r['ood_acc']:.3f}")
    return r


def plot_histories(histories: dict[str, dict], title: str = "") -> None:
    fig, axs = plt.subplots(1, 3, figsize=(16, 4))
    for (name, h), color in zip(histories.items(), plt.cm.tab10.colors):
        epochs = range(1, len(h["train_loss"]) + 1)
        axs[0].plot(epochs, h["train_loss"], color=color, label=name)
        axs[1].plot(epochs, h["train_acc"], "--", color=color, alpha=0.7)
        axs[1].plot(epochs, h["val_acc"], color=color, label=name)
        axs[2].plot(epochs, h["val_bal_acc"], color=color, label=name)
        best = h["best_epoch"]
        axs[2].scatter(best, h["val_bal_acc"][best - 1], marker="*", s=150, color=color, edgecolor="black", zorder=5)
    titles = ["training loss", "accuracy (dashed = train, solid = ID val)", "ID val balanced accuracy (star = kept epoch)"]
    for ax, t in zip(axs, titles):
        ax.set_title(t)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Sanity check of your implementation, with a tiny model and 2,000 training patches (a few seconds).
torch.manual_seed(seed)
check_model = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(3 * 64 * 64, len(CLASSES))).to(DEVICE)
check_train = DataLoader(TensorDataset(X_train[:2_000], y_train[:2_000]), batch_size=128, shuffle=True)
check_val = DataLoader(TensorDataset(X_id_val[:1_000], y_id_val[:1_000]), batch_size=512)
check_history = fit(check_model, check_train, check_val, optim.Adam(check_model.parameters(), lr=1e-3), 3, rgb,
                    verbose=False)

preds, labels = predict(check_model, check_val, rgb)
assert preds.shape == labels.shape == (1_000,), "predict must return one prediction and one label per patch"
assert np.array_equal(preds, predict(check_model, check_val, rgb)[0]), "predict must put the model in evaluation mode"
for key in ["train_loss", "train_acc", "val_acc", "val_bal_acc"]:
    assert len(check_history[key]) == 3, f'history["{key}"] must have one value per epoch'
assert check_history["train_time"] is not None, 'history["train_time"] is missing'
best = check_history["best_epoch"]
assert best in (1, 2, 3) and check_history["val_bal_acc"][best - 1] == max(check_history["val_bal_acc"]), \
    "best_epoch must be the epoch with the best ID validation balanced accuracy, counting from 1"
assert abs(evaluate(check_model, check_val, rgb)["bal_acc"] - check_history["val_bal_acc"][best - 1]) < 1e-6, \
    "fit must restore the weights of the best epoch"
first_batch = lambda: next(iter(make_loader("train", batch_size=8, shuffle=True)))[1]
assert torch.equal(first_batch(), first_batch()), "the shuffled order must be reproducible (use shuffle_seed)"
print("All checks passed.")

---

## Experiment 1: Is the Convolutional Structure Necessary?

A CNN builds three assumptions into the network:
- **locality**: each unit of a convolutional layer looks only at a small neighborhood of the previous layer;
- **weight sharing**: the same filter is applied at every position, so a pattern is detected wherever it appears;
- **hierarchy**: deeper layers combine the patterns of earlier ones, over larger and larger regions.

An MLP applied to the flattened pixels has none of them: each of the 64×64×3 = 12,288 inputs has its own weight in
every unit of the first layer, and any spatial relationship must be learned from the data. In Lecture 18, the same
MLP that worked on Fashion-MNIST (centered, aligned objects) failed on CIFAR-10 (objects anywhere in the image),
while a CNN with 5 times fewer parameters did much better. Are tissue patches closer to the first case or to the
second?

This comparison also illustrates that **the number of parameters does not determine generalization**: the first
layer of the MLP alone has about 65 times more parameters than the whole SmallCNN.

> **Question:**
>
> Does the spatial inductive bias of a CNN improve in-distribution performance *and* generalization to a new
> clinical center, compared with an MLP with many more parameters trained under the same conditions?

> **Hypothesis:**
>
> A tissue class is defined by local patterns (the shape, size and density of nuclei and fibers, the arrangement of
> glands and cells), which can appear at any position of the patch and are the same in every center. A SmallCNN, which
> detects local patterns anywhere with shared filters, will therefore reach a **higher balanced accuracy than the MLP
> on both the ID and the OOD validation splits**, and a **smaller ID–OOD gap**, despite having more than 60 times fewer
> parameters.
> The hypothesis is contradicted if the MLP matches or beats the CNN on the OOD validation balanced accuracy, or if the
> gap of the CNN is not smaller than the gap of the MLP.

> **Baseline:**
>
> A classifier that always predicts the same class has balanced accuracy 1/9 ≈ 0.11.

In [ ]:
class MLP(nn.Module):
    # Fully connected network on the flattened patch: no spatial inductive bias.

    def __init__(self, input_size: int, hidden_sizes: list[int], n_classes: int = 9, dropout: float = 0.0) -> None:
        super().__init__()
        # TODO: build the network with nn.Sequential:
        #   Flatten -> [Linear -> ReLU -> Dropout] for each hidden size -> Linear(last hidden size, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: x has shape (batch, 3, 64, 64); return the logits, with shape (batch, n_classes).
        return x


class SmallCNN(nn.Module):
    # Small convolutional network: locality, weight sharing and hierarchy.

    def __init__(self, channels: tuple[int, ...] = (32, 64, 128), n_classes: int = 9) -> None:
        super().__init__()
        # TODO: build the network with the following structure:
        #   one block per entry of `channels`, each: Conv2d(kernel_size=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        #   (the first block receives the 3 color channels)
        #   then: AdaptiveAvgPool2d(1) -> Flatten -> Linear(channels[-1], n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: return the logits, with shape (batch, n_classes).
        return x

In [ ]:
mlp = MLP(input_size=3 * 64 * 64, hidden_sizes=[512, 256])
cnn = SmallCNN()

print(f"MLP parameters      : {count_params(mlp):>12,}")
print(f"SmallCNN parameters : {count_params(cnn):>12,}")
print()
print("Both models receive the same standardized 64x64x3 patches and are trained with the same loop.")

In [ ]:
train_loader = make_loader("train", batch_size=128, shuffle=True)
id_val_loader = make_loader("id_val")

# Hint: the comparison is fair if both models are trained under the same conditions (same optimizer, learning rate
# and number of epochs), or if each one is given the chance to perform at its best (tuning each one separately on the
# ID validation split). If the CNN beats the MLP but they were trained very differently, it is hard to argue that the
# architecture is the cause.

learning_rate = ...
n_epochs = ...
dropout = ...

In [ ]:
torch.manual_seed(seed)
mlp = MLP(input_size=3 * 64 * 64, hidden_sizes=[512, 256], dropout=dropout).to(DEVICE)
mlp_optimizer = ...

mlp_history = fit(mlp, train_loader, id_val_loader, mlp_optimizer, n_epochs, rgb)
report("MLP (Exp. 1)", mlp, rgb, mlp_history);

In [ ]:
torch.manual_seed(seed)
cnn = SmallCNN().to(DEVICE)
cnn_optimizer = ...

cnn_history = fit(cnn, train_loader, id_val_loader, cnn_optimizer, n_epochs, rgb)
report("SmallCNN (Exp. 1)", cnn, rgb, cnn_history);

In [ ]:
# Plot the training histories of both models (plot_histories) and analyze the results. Examples of figures:
# - the recall of each class on the ID and on the OOD validation splits, and the confusion matrices
#   (sklearn.metrics.confusion_matrix with the outputs of predict());
# - the first-layer weights of the MLP, reshaped to 64x64x3 images, and the first-layer filters of the CNN;
# - a 2D projection (PCA or t-SNE) of the last hidden representation of each model, colored by class and by center
#   (id_val vs. ood_val).

### Design justification

> **Question.** (0.5 pt)
>
> Why does comparing an MLP and a SmallCNN, trained from scratch on the same patches, with the same loop and the same
> evaluation splits, isolate the contribution of the convolutional inductive bias? Which factors did you keep fixed,
> and which differences between the two models (besides the inductive bias) could still explain the results?
>
> Which metrics did you use to test the hypothesis, and why do they answer the question?

*YOUR ANSWER HERE*

### Discussion

> **Question.** (1.0 pt)
>
> Discuss the results of the MLP and of the SmallCNN. Address the following points:
>
> Do the results support or contradict the hypothesis? Refer to specific numbers (ID and OOD balanced accuracy, gap).
>
> Compare the learning curves of the two models. Did the MLP use its extra parameters to fit the training set better?
> What does that say about the relation between the number of parameters and generalization?
>
> Which classes lose the most on the new center, for each model? What do the first-layer weights of the MLP and the
> filters of the CNN look like, and does what each model learned help to explain its behavior on the new center?

*YOUR ANSWER HERE*

---

## Experiment 2: Does the Network Know That a Patch Has No "Up"?

A tissue patch has no canonical orientation: rotated by 90°, 180° or 270°, or mirrored, a patch of smooth muscle is
still smooth muscle. These 8 transformations (4 rotations, each with or without a mirror) are **symmetries of the
task**.

Neither architecture has this symmetry built in. An MLP ties each weight to a pixel position, and the filters of a CNN
are oriented: a filter that detects vertical edges does not respond to horizontal ones, so a rotated patch produces
different feature maps. In Lecture 18, a CNN was more tolerant to shifts than an MLP, but it became robust to them
only after **data augmentation** showed it shifted images during training: *convolution gives the network the ability
to detect a pattern anywhere; augmentation teaches it to use that ability*.

The code below provides the 8 symmetries as functions on a batch of images, and an augmentation that applies one of
them, chosen at random, to each training patch.

> **Question:**
>
> Are the models of Experiment 1 invariant to rotations and mirroring of the patch? Does training with these
> symmetries as data augmentation make the MLP and the CNN invariant, and does it reduce the ID–OOD gap?

### Hypothesis

> **Hypothesis.** (0.5 pt)
>
> *State a falsifiable hypothesis. It should predict (i) how the balanced accuracy of the Experiment 1 models changes
> when the evaluation patches are rotated or mirrored, and whether the MLP and the CNN differ, and (ii) the direction
> of the change in OOD balanced accuracy and in the ID–OOD gap when each architecture is trained with the symmetries
> as augmentation. Relate your prediction to the structural properties of each architecture.*

*YOUR ANSWER HERE*

In [ ]:
# The 8 symmetries of the square, as functions on a batch of images with shape (batch, channels, height, width).
SYMMETRIES = {
    f"{'mirror + ' if mirror else ''}rot {90 * k}": (
        lambda x, k=k, mirror=mirror: torch.rot90(x.flip(3) if mirror else x, k, dims=(2, 3))
    )
    for mirror in (False, True)
    for k in range(4)
}


def random_symmetry(x: torch.Tensor) -> torch.Tensor:
    # Augmentation: applies one of the 8 symmetries, chosen at random for each image, to a batch (N, C, H, W).
    mirror = torch.rand(len(x), device=x.device) < 0.5
    x = torch.where(mirror[:, None, None, None], x.flip(3), x)
    k = torch.randint(4, (len(x),), device=x.device)
    out = x.clone()
    for r in range(1, 4):
        out[k == r] = torch.rot90(x[k == r], r, dims=(2, 3))
    return out


# One OOD validation patch under the 8 symmetries
X, y = SPLITS["ood_val"]
patch = X[:1].float().div(255)
fig, axs = plt.subplots(1, len(SYMMETRIES), figsize=(16, 2.4))
for ax, (name, fn) in zip(axs, SYMMETRIES.items()):
    ax.imshow(fn(patch)[0].permute(1, 2, 0).numpy())
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle(f"The 8 symmetries of one patch ({CLASSES[y[0].item()]})", fontsize=12)
plt.show()

In [ ]:
# Part (i): evaluate the models of Experiment 1, WITHOUT retraining, on the OOD validation split transformed by each
# of the 8 symmetries. Hint: evaluate(model, make_loader("ood_val"), rgb, input_fn=SYMMETRIES[name]).

In [ ]:
# Part (ii): train a new MLP and a new SmallCNN, from scratch, with the symmetries as augmentation of the TRAINING
# split: fit(..., preprocess=rgb, augment=random_symmetry). Keep everything else as in Experiment 1. Then evaluate
# them with report() and under the 8 symmetries, as in part (i).

In [ ]:
# Compare the four models: learning curves, metrics under each symmetry, ID and OOD balanced accuracy, gap.

### Design justification

> **Question.** (0.5 pt)
>
> Explain your experiment configuration: what remained the same as in Experiment 1 and what changed. Why does
> evaluating a model on transformed copies of the same patches measure its invariance? Why is the augmentation the only
> difference between the two trainings of each architecture? How does your experiment answer the question?

*YOUR ANSWER HERE*

### Discussion

> **Question.** (1.0 pt)
>
> Discuss the results of the symmetry experiment. Address the following points:
>
> Were the models of Experiment 1 invariant to the symmetries? Did the MLP and the CNN behave differently? Refer to
> specific numbers.
>
> Did augmentation make each architecture invariant? Did it help the CNN and the MLP equally? Relate your answer to the
> sentence from Lecture 18 quoted above.
>
> Did augmentation change the ID–OOD gap? Augmentation also acts as a regularizer, because the model rarely sees
> exactly the same image twice. Can you attribute the change in OOD performance to the invariance itself, rather than
> to less overfitting?

*YOUR ANSWER HERE*

---

## Experiment 3: Is the Stain Color a Shortcut?

Each center has its own staining protocol and scanner, so the colors of a patch partly encode **where** it was
acquired. A network trained on a single center may learn to recognize a tissue by the exact shades it has in that
center, a **shortcut** that fits the training data but does not transfer to a center whose stains look different,
even if the morphology of the tissue is the same.

There are two opposite ways to prevent a network from relying on the exact colors:
- **remove them**: train and evaluate on grayscale patches. The network cannot use color at all, but it also loses
  the difference between the purple of hematoxylin (nuclei) and the pink of eosin (cytoplasm, fibers), which a
  pathologist uses;
- **randomize them**: keep the colors, but change the amount of each stain at random in every training patch
  (**stain augmentation**, Tellez et al., 2019), so that no exact shade is reliably associated with a class.

`stain_jitter` below converts each RGB patch into the concentrations of hematoxylin, eosin and DAB (the color
deconvolution of `skimage.color.rgb2hed`), multiplies each concentration by a random factor in [1 − `scale`,
1 + `scale`], adds a random shift in [−`shift`, `shift`], and converts back to RGB. Every patch gets its own factors.

> **Question:**
>
> Does the CNN rely on the exact stain colors of the training center? Which strategy reduces the ID–OOD gap more:
> removing color (grayscale) or randomizing the stains during training (stain augmentation)?

### Hypothesis

> **Hypothesis.** (0.5 pt)
>
> *State a falsifiable hypothesis. It should predict the direction of the change in ID balanced accuracy, OOD
> balanced accuracy and ID–OOD gap for each of the two strategies, relative to the CNN of the previous experiments, and
> which strategy is better. Consider both what color reveals about the center and what it reveals about the tissue,
> and relate your prediction to the Data Exploration.*

*YOUR ANSWER HERE*

In [ ]:
# (a) Grayscale: the same gray image in the 3 channels (so the architecture does not change), standardized with
#     statistics of the grayscale training split. Used for training AND evaluation.
def grayscale(x: torch.Tensor) -> torch.Tensor:
    gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    return gray.expand(-1, 3, -1, -1)


gray = Preprocess(color_fn=grayscale)


# (b) Stain augmentation, for the TRAINING split only (the model is evaluated with `rgb`).
RGB_FROM_HED = torch.tensor([[0.65, 0.70, 0.29],    # hematoxylin
                             [0.07, 0.99, 0.11],    # eosin
                             [0.27, 0.57, 0.78]])   # DAB      (the matrix of skimage.color.rgb2hed)
HED_FROM_RGB = torch.linalg.inv(RGB_FROM_HED)


def stain_jitter(x: torch.Tensor, scale: float = 0.2, shift: float = 0.05) -> torch.Tensor:
    # Random change of the amount of each stain, with its own factors for each image of a batch (N, 3, H, W) in [0, 1].
    optical_density = -torch.log(x.clamp(min=1e-6)).permute(0, 2, 3, 1)           # (N, H, W, 3)
    stains = optical_density @ HED_FROM_RGB.to(x.device)                         # concentrations of H, E and DAB
    factors = 1 + scale * (2 * torch.rand(len(x), 1, 1, 3, device=x.device) - 1)
    shifts = shift * (2 * torch.rand(len(x), 1, 1, 3, device=x.device) - 1)
    stains = stains * factors + shifts
    return torch.exp(-(stains @ RGB_FROM_HED.to(x.device))).clamp(0, 1).permute(0, 3, 1, 2)


# Choose the strength of the augmentation and justify it: the default values are only a starting point.
# Example of use: fit(..., preprocess=rgb, augment=stain_jitter). If you start from Experiment 2, combine both:
# augment=lambda x: random_symmetry(stain_jitter(x)).

In [ ]:
# The same training patches: original, grayscale and four draws of stain augmentation
X, y = SPLITS["train"]
patches = X[:6].float().div(255)
torch.manual_seed(seed)
rows = {"original": patches, "grayscale": grayscale(patches)}
rows.update({f"stain jitter {k + 1}": stain_jitter(patches) for k in range(4)})

fig, axs = plt.subplots(len(rows), len(patches), figsize=(1.8 * len(patches), 1.9 * len(rows)), constrained_layout=True)
for i, (name, batch) in enumerate(rows.items()):
    for j in range(len(patches)):
        axs[i, j].imshow(batch[j].permute(1, 2, 0).numpy())
        axs[i, j].set_xticks([])
        axs[i, j].set_yticks([])
        if i == 0:
            axs[i, j].set_title(CLASSES[y[j].item()], fontsize=7)
    axs[i, 0].set_ylabel(name, fontsize=9)
plt.show()

In [ ]:
# Your code here: start from the SmallCNN configuration of Experiment 1 or 2 (state which one) and change ONLY the
# color processing, so that the effect of color is isolated. Train the grayscale CNN and the stain-augmented CNN,
# evaluate them with report() (gray for the grayscale CNN, rgb for the stain-augmented one) and analyze.

### Design justification

> **Question.** (0.5 pt)
>
> Explain your experiment configuration: which CNN you started from, what remained the same and what changed. Why do
> the two strategies test whether the CNN relies on the exact colors, and what does each of them assume about the data?
> How does your experiment answer the question?

*YOUR ANSWER HERE*

### Discussion

> **Question.** (1.0 pt)
>
> Discuss the results of removing and of randomizing the colors. Address the following points:
>
> Does each strategy improve or degrade the ID metrics? And the OOD metrics? Was this consistent with your hypothesis?
>
> Which classes gain or lose the most with each strategy? Can you attribute the results uniquely to color? Think
> about what else grayscale and stain augmentation change.
>
> How do the results connect with the Data Exploration you conducted?

*YOUR ANSWER HERE*

---

## Experiment 4: Is the Gap Real, and Where Does It Come From?

Experiments 1-3 compared models trained with a single seed, and measured a single number for the gap. Two questions
remain before choosing a final model.

**Is a difference between two models larger than the noise of training?** The seed changes the initialization, the
order of the mini-batches and the augmentations. On distribution-shift benchmarks, the OOD performance of the same
configuration can change a lot from one seed to another: on Camelyon17, another histopathology task, the WILDS
benchmark reports 70.3 ± 6.4% OOD accuracy over 10 seeds. A difference between two models smaller than this
variation is not evidence of anything.

**Where does the gap come from?** The new center differs in the appearance of the patches (*covariate shift*), but
it may also have different proportions of the 9 classes (*label shift*). The two effects can be separated. Let
$r_{\text{ID}}(c)$ be the recall of class $c$ on `id_val`, and $\pi_{\text{OOD}}(c)$ the proportion of class $c$ in
`ood_val`. If only the proportions changed, and each class were recognized as well as in the ID validation, the OOD
accuracy would be

$$
\text{acc}_{\text{prop}} = \sum_{c=0}^{8} \pi_{\text{OOD}}(c)\, r_{\text{ID}}(c),
$$

and the accuracy gap splits into two terms:

$$
\underbrace{\text{acc}_{\text{ID}} - \text{acc}_{\text{OOD}}}_{\text{total gap}}
= \underbrace{\text{acc}_{\text{ID}} - \text{acc}_{\text{prop}}}_{\text{change of proportions}}
+ \underbrace{\text{acc}_{\text{prop}} - \text{acc}_{\text{OOD}}}_{\text{change of appearance}} .
$$

Steps:
1. **Seeds.** Retrain the MLP and the SmallCNN of Experiment 1, and the best variant of Experiment 2 and of
   Experiment 3 (chosen by ID validation balanced accuracy), with **at least 3 seeds** each (the seed used before
   counts as one). Report the mean and the standard deviation of the ID balanced accuracy, of the OOD balanced
   accuracy and of the gap.
2. **Decomposition.** Implement `gap_decomposition` and apply it to the SmallCNN of Experiment 1 and to your best
   variant.
3. **Final model.** Choose a final model using **only the ID validation**: one of the configurations above, or a new
   one that combines what worked (architecture, augmentations, the regularization tools of Lecture 18 such as dropout
   and weight decay, more epochs; you may also fine-tune a pretrained ResNet-18 from `torchvision.models`). Report its
   number of parameters and its training time, and evaluate it **once** on `ood_test`.

> **Question:**
>
> Are the differences between the models of Experiments 1-3 larger than the variation caused by the random seed? And
> is the ID–OOD gap of the CNN caused mainly by the change of class proportions or by the change of appearance of the
> patches?

### Hypothesis

> **Hypothesis.** (0.5 pt)
>
> *State a falsifiable hypothesis that answers both parts of the question: which of the comparisons of Experiments 1-3
> you expect to survive the variation between seeds, and which of the two terms of the decomposition you expect to
> dominate the gap of the CNN. Base the second prediction on your Data Exploration.*

*YOUR ANSWER HERE*

In [ ]:
def run_seeds(build_model, build_optimizer, n_epochs: int, preprocess: Preprocess, augment=None,
              seeds=(42, 43, 44)) -> pd.DataFrame:
    # Trains the same configuration with several seeds and returns one row of metrics per seed.
    rows = []
    for s in seeds:
        torch.manual_seed(s)
        model = build_model().to(DEVICE)
        train = make_loader("train", batch_size=128, shuffle=True, shuffle_seed=s)
        fit(model, train, make_loader("id_val"), build_optimizer(model), n_epochs, preprocess, augment, verbose=False)
        id_m = evaluate(model, make_loader("id_val"), preprocess)
        ood_m = evaluate(model, make_loader("ood_val"), preprocess)
        rows.append({"seed": s, "id_bal_acc": id_m["bal_acc"], "ood_bal_acc": ood_m["bal_acc"],
                     "gap_bal_acc": id_m["bal_acc"] - ood_m["bal_acc"]})
        print(f"seed {s}: ID bal. acc {id_m['bal_acc']:.3f} | OOD bal. acc {ood_m['bal_acc']:.3f}")
    return pd.DataFrame(rows)


# Example (adapt to your configurations):
# seed_results = {
#     "SmallCNN (Exp. 1)": run_seeds(lambda: SmallCNN(), lambda m: optim.Adam(m.parameters(), lr=learning_rate),
#                                    n_epochs, rgb),
#     ...
# }
# seed_summary = pd.DataFrame({name: df.drop(columns="seed").agg(["mean", "std"]).unstack()
#                              for name, df in seed_results.items()}).T
# display(seed_summary)

In [ ]:
def gap_decomposition(model: nn.Module, preprocess: Preprocess) -> dict[str, float]:
    # Splits the ID-OOD accuracy gap into a change of class proportions and a change of appearance.
    # TODO: with predict() on id_val and on ood_val, compute
    #   - acc_id and acc_ood, the accuracies on id_val and on ood_val;
    #   - recall_id[c], the recall of each class on id_val, and prop_ood[c], the proportion of each class in ood_val;
    #   - acc_prop = sum over c of prop_ood[c] * recall_id[c];
    # and return the total gap and its two terms.
    return {"total_gap": ..., "proportions": ..., "appearance": ...}

In [ ]:
# Your code here: seeds, decomposition, and the training of your final model (use report() to add it to RESULTS).

final_name = ...   # the name of your final model in RESULTS

In [ ]:
# OOD test: run this cell only once, after all decisions are made. The SmallCNN of Experiment 1 is evaluated as a
# reference.
for name in dict.fromkeys([final_name, "SmallCNN (Exp. 1)"]):
    model, preprocess = MODELS[name]
    test = evaluate(model, make_loader("ood_test"), preprocess)
    RESULTS[name].update(test_acc=test["acc"], test_bal_acc=test["bal_acc"])
    print(f"{name}: OOD test acc {test['acc']:.3f} | OOD test bal. acc {test['bal_acc']:.3f}")

### Discussion: seeds and decomposition

> **Question.** (1.0 pt)
>
> Do the results support or contradict your hypothesis? Refer to specific numbers.
>
> Which conclusions of Experiments 1-3 survive the variation between seeds, and which should be revised?
>
> How is the gap of the CNN divided between the change of proportions and the change of appearance? What does the
> decomposition say about the difference between the accuracy and the balanced accuracy, and about which kind of
> intervention (architecture, augmentation, color) can reduce each term?

*YOUR ANSWER HERE*

### Discussion: final model

> **Question.** (1.0 pt)
>
> What is your final model, how many parameters does it have, and how long did it take to train? Compare it with the
> previous models and draw a link with the results.
>
> How does its performance on `ood_test` compare with its performance on `ood_val`, and with the SmallCNN of
> Experiment 1? Did the gap shrink?
>
> If your results are positive, can you attribute with certainty which change contributed most? Conversely, if they
> are negative, provide an alternative explanation that is consistent with the failure.

*YOUR ANSWER HERE*

---

## Summary

Complete the table below with the results of your experiments. The seed column comes from Experiment 4, and the
test columns are filled in only for the two models evaluated on `ood_test`.

| Model | Parameters | Training time (s) | ID val bal. acc | OOD val bal. acc | Gap (bal. acc) | OOD val bal. acc over seeds (mean ± std) | OOD test bal. acc | OOD test acc |
|---|---|---|---|---|---|---|---|---|
| MLP (Exp. 1) | | | | | | | – | – |
| SmallCNN (Exp. 1) | | | | | | | | |
| MLP + symmetries (Exp. 2) | | | | | | | – | – |
| SmallCNN + symmetries (Exp. 2) | | | | | | | – | – |
| SmallCNN, grayscale (Exp. 3) | | | | | | | – | – |
| SmallCNN + stain augmentation (Exp. 3) | | | | | | | – | – |
| Final model (Exp. 4): *name* | | | | | | | | |

*Gap = ID val balanced accuracy − OOD val balanced accuracy. Leave the seed column empty for the configurations you
did not retrain.*

---

## References

- J. N. Kather et al. *Predicting survival from colorectal cancer histology slides using deep learning: A
  retrospective multicenter study.* PLOS Medicine, 2019.
- J. Yang et al. *MedMNIST v2 - A large-scale lightweight benchmark for 2D and 3D biomedical image classification.*
  Scientific Data, 2023.
- D. Tellez et al. *Quantifying the effects of data augmentation and stain color normalization in convolutional
  neural networks for computational pathology.* Medical Image Analysis, 2019.
- P. W. Koh et al. *WILDS: A Benchmark of in-the-Wild Distribution Shifts.* ICML, 2021.

---
### Declaração de Uso de IA generativa

Conforme a diretriz 4, declaramos que utilizamos o modelo Claude Opus 5.5 (Anthropic), por meio da extensão Claude Code para o VS Code, como assistente de implementação durante a realização desta tarefa.

Os prompts empregados foram, em sua maioria, dos seguintes tipos:

- Correção de bugs e de problemas de exibição;
- Melhoria da legibilidade dos gráficos e da organização do código;
- Verificação de consistência entre os números da tabela do Summary e as saídas do notebook;
- Correção gramatical pontual nas respostas.

As respostas às questões foram redigidas pelos integrantes do grupo, e todo o código gerado foi revisado por eles antes de ser incorporado.

---

## Assignment Submission

This notebook must be filled with your solution and submitted on the Assignment's entry at our Google Classroom page.

Before submitting, verify that:
- The notebook runs end-to-end without errors (use **Run All** after restarting the kernel).
- All plots are rendered in the output.
- All written sections (Hypothesis, Design, Discussion) are filled in.
- The Summary table is completed.
- AI usage declaration and prompts are included, if applicable.

**Important: Only one member of the group should submit.**

### Policy for late submissions

You are NOT encouraged to submit after the deadline; however, in this case the following penalties apply:

- 25% deduction for 1-day late submission.
- 50% deduction for 2-day late submission.
- 75% deduction for 3-day late submission.